# Adapting REMIND to CLIP**Alexie Linardatos**This document covers one thing: the REMIND paper, why it is the right foundation for the replaystorage question in my thesis, what breaks when it is moved from a CNN to CLIP, and how it isimplemented in my codebase today.Contents:1. What REMIND does2. Why REMIND is the right paper for this work3. What breaks when you move it to CLIP4. How it is implemented in my code5. Current status and results

## 1. What REMIND doesREMIND Your Neural Network to Prevent Catastrophic Forgetting. Hayes, Kafle, Shrestha, Acharya andKanan, ECCV 2020. arXiv 1910.02509. REMIND stands for REplay using Memory INDexing.The setting is streaming class-incremental learning on a ResNet-18: one sample at a time, ImageNetand CORe50, a single domain throughout.### 1.1 Three mechanisms**Store mid-network activations rather than images or final embeddings.** Tap the network partwayup and store what comes out. A stored final embedding is a dead constant and gives up the visiontower entirely; a stored image is expensive. A mid-network activation is the compromise: thereplayed sample still flows through every layer above the tap, so it carries a real gradient.**Compress with product quantization.** Split each D-dimensional vector into m contiguoussubvectors, learn a 256-entry codebook per subspace by k-means, and store one byte per subspace. A768-dimensional vector at m=32 becomes 32 bytes, a 96x compression. This is what makes mid-networkreplay cheaper than pixel replay rather than more expensive.**Freeze everything below the tap after the first task.** This is the mechanism that matters mostfor my work. The layers that produced the stored activations never change again, so the storedactivations cannot go stale. Drift is eliminated by construction rather than corrected after thefact.### 1.2 Why it works on a CNNThree properties of REMIND's original setting make all of this effective:| Property | Why it helps ||---|---|| Post-ReLU feature maps are sparse and highly redundant | effective dimensionality is far below nominal, so aggressive quantization is close to lossless || A CNN has a genuine low-to-high hierarchy | early layers are generic edge and texture detectors, so freezing them costs little and the split point is principled || The stream is a single domain | the codebook is fitted once on the first task and reused, which is sound when later data comes from the same distribution |Every one of those three is false for CLIP on MTIL. That is section 3.

## 2. Why REMIND is the right paper for this work### 2.1 The requirement my own results producedMy feature replay experiments store the final 512-d CLIP embedding. That drops storage from 6.6 GBto 11 MB but loses 1.38 points of Last, and roughly two thirds of that loss is a single task:Aircraft, which is task 0, stored first and exposed to ten subsequent task boundaries. The damagescales with how long a task has been sitting in the buffer, which is the signature of representationdrift.I then built a ladder of drift-correction methods: SDC kernel drift, label propagation, and learnedlinear and MLP maps, crossed with image, text and both anchor types. All four variants landed within0.24 points of each other on Last. Correction does not recover the loss.That produces two requirements, and they are jointly restrictive:> **(a)** Prevent drift rather than correct it, because correction was measured and found weak.>> **(b)** Keep the replayed sample on a live gradient path, because a stored final embedding gives> up the image tower entirely.### 2.2 Every alternative family fails one of the two| Family | Representative work | Prevents drift | Live gradient ||---|---|---|---|| Drift compensation | SDC (Yu CVPR 2020), A2LP (Zhang ECCV 2020), Iscen ECCV 2020 | no, corrects after the fact | no, final-layer features || Generative latent replay | CGIL (BMVC 2024 oral) | partly, samples are redrawn but the VAE itself drifts | no, final-layer embeddings || Generative pixel replay | LoRA-Loop (2025), DGR-style methods | yes, regenerates images | yes, but reintroduces a pixel pipeline and a generator to train || Rehearsal-free | AFA (2025), DIKI, BCL | not applicable, no buffer exists | not applicable || Compressed activation replay | **REMIND**, ACAE-REMIND, SHARP | **yes, by freezing below the tap** | **yes, via the mid-network tap** |Three observations follow.First, the drift-compensation family is exactly what my adaptation ladder already implements, and Imeasured it failing. Having done that, moving to prevention is the correct next step rather thantrying a fourth estimator.Second, the rehearsal-free family answers a different question. Those methods have no buffer, sothey cannot inform a study about what a buffer should contain. They are the right competitivebaselines, but they are not something to build on here.Third, within the compressed-activation family, ACAE-REMIND (Pattern Recognition Letters 2021) andSHARP (2023) are refinements of REMIND rather than independent alternatives. ACAE-REMIND swapsproduct quantization for an auxiliary-classifier autoencoder; SHARP adds sparsity and replays onlyrecent classes. Both inherit REMIND's split-and-freeze structure. Implementing REMIND thereforeimplements the common core of the whole family, and those two become follow-up ablations rather thanseparate projects.### 2.3 The conclusionREMIND is not one option among several. It is the canonical member of the only family that satisfiesboth requirements, and porting it is what makes the resulting comparison legible to readers who knowthis literature.There is also a CLIP-specific bonus that REMIND itself could not have claimed. Freezing the lowerblocks holds half the vision tower at pre-trained CLIP weights, which should help zero-shotretention. REMIND has no zero-shot metric; MTIL does. So the same mechanism that prevents drift mayalso protect Transfer, and that is a prediction worth testing rather than an assumption.

## 3. What breaks when you move it to CLIPFive mismatches. The first four are engineering problems. The fifth is information-theoretic andcannot be engineered away.| | Mismatch | REMIND's assumption | What CLIP does instead ||---|---|---|---|| 1 | One codebook, eleven domains | single-domain stream, so fit once on task 0 and reuse | MTIL spans aircraft, textures, satellite, digits, food, so a codebook fitted on Aircraft is used to quantize MNIST || 2 | Outlier channels | variance spread evenly across dimensions | transformer residual streams carry a few channels an order of magnitude larger, which swamp the subspaces they land in || 3 | No natural seam | CNN has a real low-to-high hierarchy, so layer 4.0 is principled | every ViT block is the same full-width residual update with content-dependent attention, so there is no principled split || 4 | The classifier drifts | replays into a linear head trained continuously, no staleness there | CLIP has no head, classification is against text embeddings from an encoder that keeps moving || 5 | The bit budget | sparse low-rank CNN maps compress almost losslessly | ViT residual streams are near full-rank, so the same bits buy far less fidelity |### 3.1 Mismatch 1 in detailThe original implementation fits the codebook on task 0 only, with this reasoning written into thecode: refitting later would change what every previously stored code means, so it is deliberately aone-time event.That is correct for a shared codebook and harmless on single-domain ImageNet. On MTIL it means onecodebook fitted on Aircraft is used for ten other domains. Measured reconstruction error on my first11-task run was 0.5804, and that figure was measured on Aircraft's own tokens, meaningin-distribution. It is worse everywhere else.### 3.2 Mismatch 2 in detailProduct quantization splits a vector into contiguous subvectors and implicitly assumes variance isspread evenly across dimensions. Transformer residual streams violate this. A small number ofchannels carry very large magnitudes, and those channels concentrate into a few subspaces which 256centroids cannot cover, while the remaining subspaces are starved.The correct fix is per-channel standardisation across the token population, not per-tokenLayerNorm. LayerNorm rescales each token across its 768 dimensions but leaves the relative scalebetween channels untouched, and the between-channel scale is the part that causes the problem.### 3.3 Mismatch 4 in detailThis one has no analogue in the original paper. REMIND replays into a linear classifier head thatkeeps training, so there is no staleness on the classifier side. CLIP has no head: classification isthe stored image representation against class-name text embeddings produced by an encoder that keepsmoving.Worse, from my gradient-routing analysis, the replay cross-entropy is the only term in the wholeobjective that updates the text encoder for previously seen classes. So a correct REMIND portinherits a text-side drift problem that REMIND never had to consider, and the only thing constrainingthat drift is replay itself. This appears to be a new problem statement for latent replay invision-language models.### 3.4 Mismatch 5 in detail, the one that cannot be fixedEach product quantization subvector holds D/m dimensions coded in 8 bits, so the rate per dimensionis R = 8m/D. For a Gaussian source under squared error the distortion is sigma squared times 2 tothe power minus 2R, so the relative L2 error floor is 2 to the power minus R.| m | Dims per subvector | Bits per dim | Error floor | kB per exemplar | Buffer at 11k ||---|---|---|---|---|---|| 32 | 24 | 0.33 | 0.794 | 6.3 | 69 MB || 64 | 12 | 0.67 | 0.630 | 12.6 | 139 MB || 96 | 8 | 1.00 | 0.500 | 18.9 | 208 MB || 128 | 6 | 1.33 | 0.397 | 25.2 | 277 MB || 192 | 4 | 2.00 | 0.250 | 37.8 | 416 MB || 256 | 3 | 2.67 | 0.157 | 50.4 | 555 MB |My observed quantised-space errors at m=32 were 0.765, 0.740 and 0.487 on DTD, EuroSAT and MNIST,against a predicted floor of 0.794. The codebook is already operating at the theoretical limit forits bit budget. The values sit slightly below the isotropic floor because real data retains somestructure to exploit.Two consequences. A learned rotation, meaning OPQ, cannot rescue this, because a rotation rebalanceswhich subspaces carry variance but cannot beat a rate bound. And whitening pushes toward the bound,because making data isotropic removes exactly the structure product quantization was exploiting; itfixes load balancing and costs compressibility, and the two effects partially cancel.The deeper point is that REMIND's headline compression ratio was never transferable. It depended ona property of CNN feature maps that ViT residual streams do not have.

## 4. How it is implemented in my codeEverything lives behind one flag on the existing phase-3 trainer, replay_storage, which takes pixel,feature or remind. Pixel runs stay bit-identical to the ExRD baseline, so the comparison iscontrolled.### 4.1 Files| File | Contents ||---|---|| src/product_quantizer.py | ProductQuantizer: k-means fit, encode to uint8 codes, decode, reconstruction error. Implemented directly rather than via faiss, since Lloyd's algorithm on 24-dimensional subspaces is short and avoids a dependency the Compute Canada wheelhouse may not carry. || src/remind_buffer.py | The split forward pass, the freeze, and RemindReplayBuffer. || phase3/losses_phase3.py | compute_remind_replay_loss, the replay cross-entropy over decoded activations. || phase3/trainer_phase3.py | Calls freeze_below_layer once when replay_storage is remind. || phase3/args_phase3.py | The flags. || scripts/11task/remind_11t.sh | The 11-task launcher. LAYER and PQ_M are environment variables. |### 4.2 The split forward passThree functions in src/remind_buffer.py carve CLIP's visual transformer in half.**encode_to_layer(visual, x, layer)** runs the patch embedding and the first layer residualblocks, mirroring VisualTransformer.forward up to the split point, and returns tokens in NLD layout.For ViT-B/16 at layer 6 that is 197 tokens of 768 dimensions per image: one CLS token plus a 14 by14 grid of patch tokens.**encode_from_layer(visual, tokens, layer)** runs blocks from layer onward plus the finalprojection, so the output matches what visual.forward would have produced for the same image. Thisis the function that gives replayed samples a real gradient.**freeze_below_layer(model, layer)** sets requires_grad to False on the patch embedding and everyresidual block below the split, and returns how many parameters it froze. The trainer calls it once,at trainer_phase3.py line 216, when the storage mode is remind.### 4.3 The bufferRemindReplayBuffer mirrors the public surface of the pixel and feature buffers, so the trainer canhold any of the three interchangeably. Its memory is a dictionary keyed by task id, each entryholding a uint8 code tensor of shape (N, tokens, m) and an int64 label tensor.add_task does the following at each task boundary:1. Randomly sample the task's exemplar allocation from its training set.2. Push them through encode_to_layer to get layer-6 tokens, on the model in eval mode.3. Standardise per channel using statistics from this task's own token population.4. Fit a product quantizer on the standardised tokens.5. Encode to uint8 codes and store, alongside the codebook and the statistics.rebalance_proportional then trims every task's allocation so the total stays inside the budget, withper-class uniform allocation as the default.### 4.4 The replay losscompute_remind_replay_loss receives a batch of codes, labels and task ids from a flat dataset thatmixes tasks. It loops over the unique task ids in the batch because each task has its own classnames and its own codebook. For each task it:1. Tokenizes that task's class names through that task's prompt template and computes text   embeddings with the live model, so gradient reaches the text encoder.2. Decodes the stored codes back to tokens. Decoding is a codebook lookup and carries no gradient.3. Pushes the decoded tokens through encode_from_layer, so gradient reaches the upper blocks of the   vision tower.4. Normalises both, forms scaled logits, and takes a label-smoothed cross-entropy, weighted by   sample count and any per-task weighting.This is the point of the whole exercise. The pixel version re-encodes from raw images so gradientreaches the whole vision tower. The feature version reads a dead constant so gradient reaches onlythe text tower. This version reaches the top of the vision tower and the text tower both, while thefrozen lower blocks guarantee the stored codes never go stale.### 4.5 My two adaptationsBoth address mismatches from section 3, both are on by default, and both have ablation flags so thepublished behaviour can be recovered.**Per-task codebooks.** Each task fits and stores its own codebook beside its codes, and decodes withthe codebook it was encoded by. A later fit therefore cannot change the meaning of an earlier code,which dissolves the constraint that motivated REMIND's fit-once policy. This addresses mismatch 1.Disabled with remind_shared_codebook.**Per-channel whitening.** Each task's token population is standardised per dimension beforequantization and un-standardised on decode. Statistics are stored at fp16 and rounded before use, sothe forward transform is the exact inverse of what decode applies. This addresses mismatch 2.Disabled with remind_no_whiten.I also changed the diagnostics. add_task now reports reconstruction error in two spaces: thequantised space, which measures how well the codebook covers what it was fitted on, and the originalspace, which is what the network actually receives back and is the figure comparable to runs madebefore whitening existed.### 4.6 Honest storage accountingThe nbytes method now counts the codebooks and statistics needed to decode, not only the codes.| Component | Size ||---|---|| Codes, 11,000 exemplars by 197 tokens by 32 bytes | 69.3 MB || Codebooks, one per task, 11 by 393 kB | 4.3 MB || Whitening statistics | 33.8 kB || **Total** | **73.7 MB** |Per-task codebooks add 6.3% overhead. The result is 90x smaller than the 6.62 GB pixel buffer and 6xlarger than pure feature replay at 11 MB.### 4.7 Flags| Flag | Default | Effect ||---|---|---|| replay_storage | pixel | pixel, feature or remind || remind_layer | 6 | which visual block to split at. Lower stores more general activations and frees more of the network to train; higher compresses better and freezes more. || remind_pq_m | 32 | product quantization subvectors per token. Bytes per token equals this. 768 must divide by it. || remind_shared_codebook | off | ablation: restore REMIND's fit-once codebook policy || remind_no_whiten | off | ablation: skip per-channel standardisation |

## 5. Current status and results### 5.1 The run so farOne full 11-task run has completed, using the original REMIND settings, meaning a shared codebookand no whitening. The two adaptations were implemented afterwards and are running now.| Run | Avg | Last | T[IN] | T[UT] | Storage ||---|---|---|---|---|---|| ExRD / v4 (pixel replay) | 77.17 | 86.17 | 68.37 | 69.18 | 6.6 GB || Feature replay v0 | 76.08 | 84.79 | 68.12 | 68.81 | 11 MB || REMIND l6 m32, as published | 76.00 | 84.62 | 68.55 | 68.88 | 69 MB |T[IN] is the held-out ImageNet column, the convention used internally. T[UT] is the upper triangle,the convention ZSCL, DIKI, BCL and AFA report.### 5.2 What the per-task numbers say| Task | ExRD | Feature replay | REMIND ||---|---|---|---|| **Aircraft** (task 0, oldest) | 54.52 | 44.55 | **49.02** || StanfordCars (fine-grained) | 85.54 | 84.63 | 81.93 || EuroSAT | 94.61 | 95.02 | 93.00 |The freeze works. REMIND recovers 4.5 points on the oldest and most drift-exposed task, which ismore than any drift-correction method managed, and it does so by prevention rather than correction.It then pays for it on tasks that need low-level feature adaptation, so the mean does not move. Thatis a clean stability and plasticity trade, and it is the expected consequence of mismatch 3: freezingsix ViT blocks is a much larger commitment than freezing a ResNet's early convolutions.### 5.3 The finding that matters mostREMIND matched feature replay on Last while replaying activations corrupted at 58% relativereconstruction error. If fidelity mattered, that could not have happened.This follows directly from my gradient-routing analysis. If replay's image-side gradient is largelyredundant with the two distillation terms, then it barely matters what image produces it. Thequantizer can be terrible and the result is unchanged.That reframes section 3.4 from a failure into a finding. The right response is not to chase a bettercodebook, which the rate bound forbids anyway, but to run the sweep as a measurement: m in 32, 64 and128, three parallel runs, and plot reconstruction error against Last. If Last is flat across a 2xrange of error, replay fidelity is demonstrably irrelevant under this objective.### 5.4 Open items| | Item | Cost ||---|---|---|| 1 | Full 11-task run with per-task codebooks and whitening | running || 2 | Product quantization sweep over m, to test whether fidelity matters at all | 3 runs, parallel || 3 | Split-depth sweep over layer 4, 6 and 8, currently an unjustified guess | 3 runs || 4 | Ablation grid: shared against per-task codebook, whitened against raw | 4 smoke runs, 10 minutes each || 5 | Feature augmentation on the token grid, which REMIND has and my port does not | half a day |Item 5 is worth flagging. REMIND applies random resized crop and mixup to reconstructed feature mapson every replay, which is what stops a fixed stored tensor from becoming the identical trainingsignal every epoch. My port has no feature augmentation at all. Since the 196 patch tokens lie on a14 by 14 grid, REMIND's crop has a faithful analogue: crop a sub-rectangle of the token grid, resize,and keep the CLS token. This is the one piece of REMIND's original design that is still missing.